# Modelo transferible: variables que existen en ambos sitios

El modelo guardado en `V_G1.ipynb` usa `xejz` (componente vertical) y `HUMEDAD_SUELO`, que **no existen** en la estación de João Pessoa.
Pero la Tabla IV del paper sí muestra las entradas con las que se aplicó el modelo allí: `xang`, `velU`, `velV`, densidad del aire,
humedad relativa, hora, `xmag_disc` y temperatura. Todas se obtienen del ángulo, la magnitud del viento, la temperatura y la altitud:

- `velU = mag·cos(θ)`, `velV = mag·sin(θ)` (θ en grados → radianes, **igual en ambos sitios**)
- `air_dens = 1.225·(288/(T+273))·exp(-h/8435)` con h = 2558 m (Cajicá) y 40 m (João Pessoa)
- `xmag_disc` = viento si ≥ 3.5 m/s, 0 si no; `TEMP_norm` = temperatura si ≥ 23 °C, 0 si no

Aquí se reconstruye ese modelo, se valida **cronológicamente**, se compara con variantes y se exporta.

In [1]:
from pathlib import Path
import json, math, platform, warnings
import numpy as np, pandas as pd, sklearn, joblib
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import ExtraTreesRegressor

BASE = Path.cwd(); DATA, MODELS, OUT = BASE/"data", BASE/"models", BASE/"outputs"
for d in (MODELS, OUT): d.mkdir(exist_ok=True)
SEED, R_LOAD = 123, 100.0
ALT = {"cajica": 2558.0, "jpa": 40.0}

def air_density(t, h): return 1.225 * (288.0 / (t + 273.0)) * np.exp(-h / 8435.0)

def rmsle(y, p): return float(np.sqrt(np.mean((np.log1p(np.clip(p,0,None)) - np.log1p(np.clip(y,0,None)))**2)))
def mape_nz(y, p):
    y, p = np.asarray(y), np.asarray(p); m = y > 0
    return float(np.mean(np.abs((y[m]-p[m])/y[m]))) if m.any() else np.nan
def metrics(y, p):
    mse = mean_squared_error(y, p)
    return dict(MAE=mean_absolute_error(y,p), MSE=mse, RMSE=math.sqrt(mse), R2=r2_score(y,p), RMSLE=rmsle(y,p), MAPE=mape_nz(y,p))

def site_features(df, site):
    # df: xang (grados), xmag (m/s), T (°C), HR (%), index = fecha
    th = np.deg2rad(df["xang"])
    out = pd.DataFrame(index=df.index)
    out["xang"] = df["xang"]
    out["velU"] = df["xmag"] * np.cos(th)
    out["velV"] = df["xmag"] * np.sin(th)
    out["air_dens"] = air_density(df["T"], ALT[site])
    out["HR"] = df["HR"]
    out["hour"] = df.index.hour
    out["xmag"] = df["xmag"]
    out["temp"] = df["T"]
    out["xmag_disc"] = df["xmag"].where(df["xmag"] >= 3.5, 0.0)
    out["TEMP_norm"] = df["T"].where(df["T"] >= 23, 0.0)
    return out

## 1. Cajicá (entrenamiento): mismas reglas de limpieza que el notebook original

In [2]:
COLS = ['ID','velU','velV','velW','xang','xejz','xmag','V_G1','FR_G1','P_AT','EG1','V_G2','FR_G2','P_AT2','EG2',
        'HUMEDAD_SUELO','HUMEDAD_RELATIVA','TEMPERATURA','fecha']
raw = pd.read_csv(DATA/"data_cajica_2022_crudo.csv", names=COLS, header=None, parse_dates=['fecha'])
df = raw.copy()
df.loc[df.xmag > 65, 'xmag'] = 0
df.loc[df.V_G1 > 25, 'V_G1'] = 0
df.loc[df.HUMEDAD_RELATIVA > 70, 'HUMEDAD_RELATIVA'] = df.HUMEDAD_RELATIVA.mean()
df.loc[df.HUMEDAD_RELATIVA < 30, 'HUMEDAD_RELATIVA'] = df.HUMEDAD_RELATIVA.mean()
real = df.drop_duplicates('fecha', keep='last').sort_values('fecha').set_index('fecha')
grid = real.asfreq('60s', method='bfill'); grid['is_real'] = grid.index.isin(real.index)
tr = grid.rename(columns={'HUMEDAD_RELATIVA':'HR','TEMPERATURA':'T'})
F = site_features(tr, 'cajica')
M = pd.concat([F, tr[['V_G1','is_real']]], axis=1).dropna()
print("Cajicá: %d filas (%d reales), %s -> %s" % (len(M), M.is_real.sum(), M.index.min(), M.index.max()))

Cajicá: 90553 filas (59758 reales), 2022-06-17 22:37:00 -> 2022-08-19 19:49:00


## 2. Variantes de entrada, validación cronológica (10 folds de ventana creciente, 1 día de separación, solo observaciones reales)

In [3]:
VARIANTS = {
 "A. 8 variables del notebook (xejz, humedad suelo, date)": None,   # se toma del experimento 02
 "B. Tabla IV: xang, velU, velV, densidad, HR, hora, xmag_disc, TEMP_norm":
     ['xang','velU','velV','air_dens','HR','hour','xmag_disc','TEMP_norm'],
 "C. B sin densidad del aire":
     ['xang','velU','velV','HR','hour','xmag_disc','TEMP_norm'],
 "D. continuas, sin discretizar ni densidad":
     ['xang','velU','velV','HR','hour','xmag','temp'],
}
X, y, isr = M.drop(columns=['V_G1','is_real']), M['V_G1'], M['is_real'].values
folds = list(TimeSeriesSplit(n_splits=10, gap=1440).split(X))
mk = lambda: ExtraTreesRegressor(n_estimators=100, n_jobs=-1, random_state=SEED)

def cv(feats):
    rows = []
    for a, b in folds:
        bb = b[isr[b]]
        m = mk().fit(X.iloc[a][feats], y.iloc[a]); rows.append(metrics(y.iloc[bb], m.predict(X.iloc[bb][feats])))
    return pd.DataFrame(rows)

res, per_fold = {}, {}
for k, feats in VARIANTS.items():
    if feats is None: continue
    per_fold[k] = cv(feats); res[k] = per_fold[k].mean()
prev = pd.read_csv(OUT/"comparacion_aleatorio_vs_cronologico.csv", index_col=0).loc["Cronológico: 10 folds (obs. reales)"]
res = {list(VARIANTS)[0]: prev, **res}
comp = pd.DataFrame(res).T[['MAE','RMSE','R2','RMSLE','MAPE']]
comp.round(4).to_csv(OUT/"comparacion_variantes_cronologico.csv"); comp.round(4)

,MAE,RMSE,R2,RMSLE,MAPE
"A. 8 variables del notebook (xejz, humedad suelo, date)",0.5148,1.2681,0.3853,0.4407,0.7098
"B. Tabla IV: xang, velU, velV, densidad, HR, hora, xmag_disc, TEMP_norm",0.3801,1.2180,0.4382,0.3693,0.7355
C. B sin densidad del aire,0.3788,1.2222,0.4338,0.3699,0.7446
"D. continuas, sin discretizar ni densidad",0.3813,1.2074,0.4482,0.3681,0.7318


## 3. Modelo final (variante B, la de la Tabla IV) y exportación

Se entrena con todo Cajicá y se guarda junto a sus metadatos y a los rangos de entrenamiento.

In [4]:
FEATS_B = VARIANTS["B. Tabla IV: xang, velU, velV, densidad, HR, hora, xmag_disc, TEMP_norm"]
FEATS_C = VARIANTS["C. B sin densidad del aire"]; FEATS_D = VARIANTS["D. continuas, sin discretizar ni densidad"]
models = {n: mk().fit(X[f], y) for n, f in (("B", FEATS_B), ("C", FEATS_C), ("D", FEATS_D))}
joblib.dump(models["B"], MODELS/"et_tablaIV_vrms.joblib", compress=3)
meta = {"target": "V_G1 (V_RMS)", "features": FEATS_B, "angle": "grados; velU=mag*cos, velV=mag*sin",
        "altitude_train_m": ALT["cajica"], "train_period": [str(M.index.min()), str(M.index.max())],
        "cv_chronological": {k: round(float(v), 4) for k, v in res[list(VARIANTS)[1]].items()},
        "ranges_p1_p99": {c: [float(X[c].quantile(.01)), float(X[c].quantile(.99))] for c in FEATS_B},
        "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__}, "seed": SEED}
(MODELS/"et_tablaIV_vrms_meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
assert np.allclose(joblib.load(MODELS/"et_tablaIV_vrms.joblib").predict(X[FEATS_B].iloc[:300]), models["B"].predict(X[FEATS_B].iloc[:300]))
print("modelo exportado y verificado")

modelo exportado y verificado


## 4. João Pessoa: construir las mismas entradas

Columnas del volcado de la estación (sin cabecera): 2 = temperatura, 4 = humedad relativa, 17 = ángulo del viento, 19 = velocidad,
33 = fecha. **Supuesto a confirmar:** la columna 19 está en km/h (las columnas 20 y 21 parecen mph y m/s), por eso `mag = col19/3.6`.

In [5]:
j = pd.read_csv(DATA/"estacion_jpa_2025_crudo.csv", sep="\t", header=None, na_values=["NULL"], low_memory=False)
jp = pd.DataFrame({'fecha': pd.to_datetime(j[33], errors='coerce'), 'T': pd.to_numeric(j[2], errors='coerce'),
                   'HR': pd.to_numeric(j[4], errors='coerce'), 'xang': pd.to_numeric(j[17], errors='coerce'),
                   'xmag': pd.to_numeric(j[19], errors='coerce') / 3.6}).dropna().drop_duplicates('fecha').sort_values('fecha').set_index('fecha')
FJ = site_features(jp, 'jpa')
dt_h = jp.index.to_series().diff().median().total_seconds() / 3600
print("JPA: %d registros, %s -> %s, intervalo mediano %.0f min" % (len(jp), jp.index.min(), jp.index.max(), dt_h*60))
print("viento medio JPA %.2f m/s (paper: 3.41) | Cajicá %.2f m/s" % (jp.xmag.mean(), M.xmag.mean()))
print("JPA con viento >= 3.5 m/s: %.1f %% | Cajicá: %.1f %%" % (100*(FJ.xmag_disc>0).mean(), 100*(M.xmag_disc>0).mean()))
print("JPA con T >= 23 C: %.1f %% | Cajicá: %.1f %%" % (100*(FJ.TEMP_norm>0).mean(), 100*(M.TEMP_norm>0).mean()))
FJ.head(3).round(3)

JPA: 9229 registros, 2025-03-27 17:05:00 -> 2025-09-05 13:51:00, intervalo mediano 10 min
viento medio JPA 1.66 m/s (paper: 3.41) | Cajicá 1.83 m/s
JPA con viento >= 3.5 m/s: 11.6 % | Cajicá: 10.7 %
JPA con T >= 23 C: 94.0 % | Cajicá: 26.9 %


,xang,velU,velV,air_dens,HR,hour,xmag,temp,xmag_disc,TEMP_norm
fecha,,,,,,,,,,
2025-03-27 17:05:00,86.0,0.00,0.000,1.191,57.0,17,0.000,21.7,0.0,0.0
2025-03-29 18:18:00,34.0,0.00,0.000,1.198,68.0,18,0.000,20.1,0.0,0.0
2025-04-07 18:28:00,275.0,0.01,-0.111,1.191,55.0,18,0.111,21.7,0.0,0.0


### Fuera de distribución (variante B, percentiles 1–99 de Cajicá)

In [6]:
rng = meta["ranges_p1_p99"]
flags = pd.DataFrame({c: (FJ[c] < rng[c][0]) | (FJ[c] > rng[c][1]) for c in FEATS_B})
ood = (100 * flags.mean()).round(1).rename("% fuera de rango")
ood_any = round(100 * flags.any(axis=1).mean(), 1)
print(ood.to_string()); print("alguna variable fuera de rango: %.1f %%" % ood_any)

xang           0.1
velU           3.5
velV           1.8
air_dens     100.0
HR            88.7
hour           0.0
xmag_disc      1.8
TEMP_norm      1.0
alguna variable fuera de rango: 100.0 %


### Proyección con incertidumbre y sensibilidad a la elección de variables

In [7]:
def project(model, feats):
    per = np.stack([t.predict(FJ[feats].values) for t in model.estimators_])
    p10, p90 = np.percentile(per, [10, 90], axis=0)
    v = per.mean(0)
    return v, p10, p90

rows, proj = {}, {}
for name, feats in (("B", FEATS_B), ("C", FEATS_C), ("D", FEATS_D)):
    v, p10, p90 = project(models[name], feats); proj[name] = (v, p10, p90)
    P = v**2 / R_LOAD; E = P * dt_h
    rows[name] = {"V_max": v.max(), "V_media": v.mean(), "V_p90_max": p90.max(), "P_max_W": P.max(),
                  "E_max_Wh_intervalo": E.max(), "E_total_Wh": E.sum(), "pct_V>1": 100*(v>1).mean()}
sens = pd.DataFrame(rows).T.round(3); sens.to_csv(OUT/"sensibilidad_proyeccion_jpa.csv"); sens

,V_max,V_media,V_p90_max,P_max_W,E_max_Wh_intervalo,E_total_Wh,pct_V>1
B,11.047,0.403,19.0,1.220,0.203,15.267,12.580
C,12.704,0.387,19.0,1.614,0.269,16.455,11.507
D,11.540,0.423,16.0,1.332,0.222,15.227,12.872


In [8]:
v, p10, p90 = proj["B"]
res_jp = jp.assign(V_pred=v, V_p10=p10, V_p90=p90, P_W=v**2/R_LOAD, E_Wh=v**2/R_LOAD*dt_h)
res_jp.to_csv(OUT/"predicciones_jpa_tablaIV.csv")
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(res_jp.index, res_jp.xmag, color='#9aa5b1', lw=.6, label='viento (m/s)'); ax2 = ax.twinx()
ax2.fill_between(res_jp.index, res_jp.V_p10, res_jp.V_p90, color='#2a6f97', alpha=.25, label='p10-p90 entre arboles')
ax2.plot(res_jp.index, res_jp.V_pred, color='#2a6f97', lw=.6, label='V_RMS predicho (V)')
ax.set_ylabel('viento (m/s)'); ax2.set_ylabel('V_RMS (V)'); ax.set_title('Joao Pessoa: proyeccion exploratoria (no validada)')
h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels(); ax.legend(h1+h2, l1+l2, loc='upper right')
plt.tight_layout(); plt.savefig(OUT/"fig_jpa_tablaIV.png", dpi=200)
summary = {"jpa_registros": int(len(jp)), "jpa_rango": [str(jp.index.min()), str(jp.index.max())], "intervalo_min": round(dt_h*60, 1),
           "viento_medio_jpa": round(float(jp.xmag.mean()), 2), "viento_medio_cajica": round(float(M.xmag.mean()), 2),
           "pct_jpa_viento_ge_3_5": round(float(100*(FJ.xmag_disc>0).mean()), 1), "pct_cajica_viento_ge_3_5": round(float(100*(M.xmag_disc>0).mean()), 1),
           "pct_jpa_T_ge_23": round(float(100*(FJ.TEMP_norm>0).mean()), 1), "pct_cajica_T_ge_23": round(float(100*(M.TEMP_norm>0).mean()), 1),
           "ood_por_variable": ood.to_dict(), "ood_alguna": ood_any, "sensibilidad": sens.to_dict(orient="index")}
(OUT/"resumen_jpa_tablaIV.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False)); print(json.dumps(summary, indent=1)[:1800])

{
 "jpa_registros": 9229,
 "jpa_rango": [
  "2025-03-27 17:05:00",
  "2025-09-05 13:51:00"
 ],
 "intervalo_min": 10.0,
 "viento_medio_jpa": 1.66,
 "viento_medio_cajica": 1.83,
 "pct_jpa_viento_ge_3_5": 11.6,
 "pct_cajica_viento_ge_3_5": 10.7,
 "pct_jpa_T_ge_23": 94.0,
 "pct_cajica_T_ge_23": 26.9,
 "ood_por_variable": {
  "xang": 0.1,
  "velU": 3.5,
  "velV": 1.8,
  "air_dens": 100.0,
  "HR": 88.7,
  "hour": 0.0,
  "xmag_disc": 1.8,
  "TEMP_norm": 1.0
 },
 "ood_alguna": 100.0,
 "sensibilidad": {
  "B": {
   "V_max": 11.047,
   "V_media": 0.403,
   "V_p90_max": 19.0,
   "P_max_W": 1.22,
   "E_max_Wh_intervalo": 0.203,
   "E_total_Wh": 15.267,
   "pct_V>1": 12.58
  },
  "C": {
   "V_max": 12.704,
   "V_media": 0.387,
   "V_p90_max": 19.0,
   "P_max_W": 1.614,
   "E_max_Wh_intervalo": 0.269,
   "E_total_Wh": 16.455,
   "pct_V>1": 11.507
  },
  "D": {
   "V_max": 11.54,
   "V_media": 0.423,
   "V_p90_max": 16.0,
   "P_max_W": 1.332,
   "E_max_Wh_intervalo": 0.222,
   "E_total_Wh": 15.227,
 

## 5. Tablas II y V definitivas (variante B, validación cronológica)

Los 20 modelos con parámetros por defecto y `random_state=123`, mismos 10 folds, puntuados solo sobre observaciones reales.

In [9]:
import time
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import (LinearRegression, Ridge, BayesianRidge, Lars, OrthogonalMatchingPursuit,
                                  HuberRegressor, ElasticNet, Lasso, LassoLars, PassiveAggressiveRegressor)
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
ALL = {
 'Extra Trees Regressor': lambda: ExtraTreesRegressor(n_estimators=100, n_jobs=-1, random_state=SEED),
 'Random Forest Regressor': lambda: RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEED),
 'CatBoost Regressor': lambda: CatBoostRegressor(verbose=0, random_state=SEED, thread_count=-1),
 'Extreme Gradient Boosting': lambda: XGBRegressor(random_state=SEED, n_jobs=-1, objective='reg:squarederror'),
 'Light Gradient Boosting Machine': lambda: LGBMRegressor(random_state=SEED, verbose=-1, n_jobs=-1),
 'Gradient Boosting Regressor': lambda: GradientBoostingRegressor(random_state=SEED),
 'K Neighbors Regressor': lambda: KNeighborsRegressor(n_jobs=-1),
 'Decision Tree Regressor': lambda: DecisionTreeRegressor(random_state=SEED),
 'Bayesian Ridge': lambda: BayesianRidge(), 'Ridge Regression': lambda: Ridge(random_state=SEED),
 'Linear Regression': lambda: LinearRegression(), 'Least Angle Regression': lambda: Lars(),
 'Orthogonal Matching Pursuit': lambda: OrthogonalMatchingPursuit(), 'Huber Regressor': lambda: HuberRegressor(),
 'Elastic Net': lambda: ElasticNet(random_state=SEED), 'Lasso Regression': lambda: Lasso(random_state=SEED),
 'AdaBoost Regressor': lambda: AdaBoostRegressor(random_state=SEED),
 'Lasso Least Angle Regression': lambda: LassoLars(), 'Dummy Regressor': lambda: DummyRegressor(),
 'Passive Aggressive Regressor': lambda: PassiveAggressiveRegressor(random_state=SEED),
}
cvB = {}
for name, make in ALL.items():
    t0 = time.time(); rows = []
    for a, b in folds:
        bb = b[isr[b]]; m = make().fit(X.iloc[a][FEATS_B], y.iloc[a]); rows.append(metrics(y.iloc[bb], m.predict(X.iloc[bb][FEATS_B])))
    d = pd.DataFrame(rows); d.attrs['tt'] = (time.time() - t0) / len(folds); cvB[name] = d
    print("%-34s R2=%.4f MAE=%.4f" % (name, d.R2.mean(), d.MAE.mean()))
t2 = pd.DataFrame({n: {**d.mean().to_dict(), 'TT': d.attrs['tt']} for n, d in cvB.items()}).T.sort_values('R2', ascending=False)[['MAE','MSE','RMSE','R2','RMSLE','MAPE','TT']]
t2.round(4).to_csv(OUT/"tabla_II_varianteB.csv"); t2.round(4)

Extra Trees Regressor              R2=0.4382 MAE=0.3801


Random Forest Regressor            R2=0.4265 MAE=0.3784


CatBoost Regressor                 R2=0.4156 MAE=0.3911


Extreme Gradient Boosting          R2=0.3634 MAE=0.3994


Light Gradient Boosting Machine    R2=0.4351 MAE=0.3709


...

Linear Regression                  R2=0.3831 MAE=0.5868
Least Angle Regression             R2=0.3489 MAE=0.6213
Orthogonal Matching Pursuit        R2=-0.0222 MAE=0.7283


Huber Regressor                    R2=0.1304 MAE=0.3926
Elastic Net                        R2=0.1970 MAE=0.6394
Lasso Regression                   R2=0.0935 MAE=0.6832


AdaBoost Regressor                 R2=0.1719 MAE=0.7083
Lasso Least Angle Regression       R2=0.0935 MAE=0.6832
Dummy Regressor                    R2=-0.0183 MAE=0.7331


Passive Aggressive Regressor       R2=0.0806 MAE=0.9615


,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT
Gradient Boosting Regressor,0.3722,1.4426,1.1861,0.4644,0.3527,0.7337,3.4178
Extra Trees Regressor,0.3801,1.5203,1.2180,0.4382,0.3693,0.7355,0.3168
Light Gradient Boosting Machine,0.3709,1.5247,1.2190,0.4351,0.3613,0.7701,0.1293
Random Forest Regressor,0.3784,1.5490,1.2285,0.4265,0.3736,0.7498,1.2174
CatBoost Regressor,0.3911,1.5821,1.2399,0.4156,0.3637,0.7571,1.8302
Ridge Regression,0.5866,1.6687,1.2780,0.3832,0.4259,0.7311,0.0046
Bayesian Ridge,0.5866,1.6689,1.2781,0.3831,0.4259,0.7314,0.0094
Linear Regression,0.5868,1.6690,1.2782,0.3831,0.4260,0.7310,0.0065
Extreme Gradient Boosting,0.3994,1.7164,1.2941,0.3634,0.3788,0.8061,0.1014
Least Angle Regression,0.6213,1.7531,1.3102,0.3489,0.4333,0.7569,0.0046


In [10]:
def f4(x): return ("-" if x < 0 else "") + "%.4f" % abs(x)
L = []
for i, (n, r) in enumerate(t2.iterrows()):
    L.append("        %s & %s \\\\" % (n, " & ".join(f4(r[c]) for c in ['MAE','MSE','RMSE','R2','RMSLE','MAPE','TT'])))
    if i == 0: L.append("        \\hline")
(OUT/"tabla_II_varianteB.tex").write_text("\n".join(L))
et = cvB['Extra Trees Regressor'][['MAE','MSE','RMSE','R2','RMSLE','MAPE']]
R = ["        %d & %s \\\\" % (i, " & ".join(f4(v) for v in r)) for i, r in et.iterrows()]
R += ["        \\hline", "        \\textbf{Mean} & " + " & ".join("\\textbf{%s}" % f4(v) for v in et.mean()) + " \\\\",
      "        \\textbf{Std} & " + " & ".join(f4(v) for v in et.std()) + " \\\\"]
(OUT/"tabla_V_varianteB.tex").write_text("\n".join(R)); et.round(4).to_csv(OUT/"tabla_V_varianteB.csv")
print((OUT/"tabla_V_varianteB.tex").read_text())

        0 & 0.2760 & 1.2649 & 1.1247 & 0.3321 & 0.3205 & 0.7437 \\
        1 & 0.3638 & 1.6409 & 1.2810 & 0.4679 & 0.3600 & 0.6991 \\
        2 & 0.3892 & 1.5441 & 1.2426 & 0.2848 & 0.3987 & 0.8026 \\
        3 & 0.3387 & 1.3202 & 1.1490 & 0.4790 & 0.3416 & 0.7469 \\
        4 & 0.4045 & 1.3527 & 1.1631 & 0.3736 & 0.3943 & 0.7139 \\
        5 & 0.2181 & 0.7690 & 0.8769 & 0.5766 & 0.2679 & 0.6869 \\
        6 & 0.3476 & 1.2626 & 1.1237 & 0.3817 & 0.3646 & 0.7599 \\
        7 & 0.3713 & 1.3252 & 1.1512 & 0.5165 & 0.3837 & 0.7933 \\
        8 & 0.5505 & 2.6457 & 1.6266 & 0.5335 & 0.4068 & 0.6280 \\
        9 & 0.5417 & 2.0780 & 1.4415 & 0.4359 & 0.4554 & 0.7809 \\
        \hline
        \textbf{Mean} & \textbf{0.3801} & \textbf{1.5203} & \textbf{1.2180} & \textbf{0.4382} & \textbf{0.3693} & \textbf{0.7355} \\
        \textbf{Std} & 0.1033 & 0.5158 & 0.2021 & 0.0937 & 0.0517 & 0.0542 \\
